# Diabetes Data Preprocessing

This notebook focuses on loading and preprocessing the Diabetes dataset. The purpose is to prepare the raw dataset for machine learning by importing the necessary libraries, locating the dataset file, loading it into a DataFrame, and checking its structure, missing values, and target variable distribution.

In [1]:
# Import regular expression and CSV modules
import re
import csv
from pathlib import Path

# Import libraries for data manipulation and analysis
import numpy as np
import pandas as pd

# Import data splitting and preprocessing tools from scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## Defining Project Paths

This step defines the paths for the base project directory, raw data folder, and processed data folder. It also creates the processed directory if it does not already exist.

In [2]:
# Set the main project directory
BASE_DIR = Path.cwd().parents[1]

# Define paths for raw and processed data
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

# Create the processed directory if needed
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Print path details to confirm everything is correct
print("Current working directory:", Path.cwd())
print("Base directory:", BASE_DIR)
print("Raw directory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Raw exists:", RAW_DIR.exists())
print("Processed exists:", PROCESSED_DIR.exists())

Current working directory: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\preprocessed
Base directory: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction
Raw directory: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\raw
Processed directory: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\processed
Raw exists: True
Processed exists: True


## Checking Raw Data Folder Contents

This step displays the available files and folders inside the raw data directory. It helps verify that the Diabetes dataset is stored in the expected location.

In [3]:
# Print all files and folders inside the raw data directory
print("Raw folder contents:")
for item in RAW_DIR.iterdir():
    print(item.name)

Raw folder contents:
Chronic_Kidney_Disease
data_source.md
Diabetes


## Locating the Diabetes Dataset File

The full file path for the Diabetes dataset is defined in this step. A check is also performed to confirm that the CSV file exists before reading it.

In [4]:
# Define the file path for the Diabetes CSV dataset
diabetes_path = RAW_DIR / "Diabetes" / "diabetes.csv"

# Print the dataset path and check whether the file exists
print("Diabetes path:", diabetes_path)
print("Diabetes exists:", diabetes_path.exists())

Diabetes path: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\raw\Diabetes\diabetes.csv
Diabetes exists: True


## Loading the Diabetes Dataset

The Diabetes dataset is loaded from the CSV file into a Pandas DataFrame. The dataset shape and first few rows are displayed to confirm successful loading.

In [5]:
# Load the Diabetes dataset from the CSV file
diabetes_df = pd.read_csv(diabetes_path)

# Display the dataset size and preview the first rows
print("Diabetes shape:", diabetes_df.shape)
display(diabetes_df.head())

Diabetes shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## Displaying Column Names

This step prints all column names in the Diabetes dataset. This helps identify the available features and the target variable.

In [6]:
# Print the column names of the Diabetes dataset
print("\nDiabetes columns:")
print(diabetes_df.columns.tolist())


Diabetes columns:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']


## Checking Data Types and Missing Values

The data types of all columns are examined in this step, along with the number of missing values. This is useful for understanding the dataset structure and identifying any preprocessing requirements.

In [7]:
# Display the data types of each column
print("Diabetes data types:")
print(diabetes_df.dtypes)

# Display the number of missing values in each column
print("\nDiabetes missing values:")
print(diabetes_df.isnull().sum().sort_values(ascending=False))

Diabetes data types:
Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

Diabetes missing values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


## Exploring the Target Variable

The target variable for the Diabetes dataset is `Outcome`. Its class distribution is displayed to understand how many records belong to each outcome category.

In [8]:
# Define the target variable for the Diabetes dataset
diabetes_target = "Outcome"

# Print the distribution of the target classes
print("\nDiabetes target distribution:")
print(diabetes_df[diabetes_target].value_counts(dropna=False))


Diabetes target distribution:
Outcome
0    500
1    268
Name: count, dtype: int64


## Cleaning the Diabetes Dataset
This step cleans the Diabetes dataset by:
- standardising column names
- checking duplicate rows
- replacing biologically impossible zero values with NaN in selected columns


In [9]:
# Make a copy so the original loaded data remains unchanged
diabetes_clean = diabetes_df.copy()

In [10]:
# Standardise column names
diabetes_clean.columns = diabetes_clean.columns.str.strip()

In [11]:
# Remove duplicate rows if any
before_rows = diabetes_clean.shape[0]
diabetes_clean = diabetes_clean.drop_duplicates()
after_rows = diabetes_clean.shape[0]

print("Rows before duplicate removal:", before_rows)
print("Rows after duplicate removal:", after_rows)
print("Duplicates removed:", before_rows - after_rows)

Rows before duplicate removal: 768
Rows after duplicate removal: 768
Duplicates removed: 0


In [14]:
# In this dataset, some medical measurement columns should not be zero
zero_as_missing_cols = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

for col in zero_as_missing_cols:
    if col in diabetes_clean.columns:
        diabetes_clean[col] = diabetes_clean[col].replace(0, np.nan)



In [15]:
# Keep Pregnancies and Outcome as they are
# DiabetesPedigreeFunction can be zero legitimately
# Age can also remain as is

# Show cleaned dataset info
print("\nDiabetes cleaned shape:", diabetes_clean.shape)



Diabetes cleaned shape: (768, 9)


In [16]:
print("\nDiabetes cleaned data types:")
print(diabetes_clean.dtypes)



Diabetes cleaned data types:
Pregnancies                   int64
Glucose                     float64
BloodPressure               float64
SkinThickness               float64
Insulin                     float64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object


In [17]:

print("\nDiabetes missing values after cleaning:")
print(diabetes_clean.isnull().sum().sort_values(ascending=False))


Diabetes missing values after cleaning:
Insulin                     374
SkinThickness               227
BloodPressure                35
BMI                          11
Glucose                       5
Pregnancies                   0
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [18]:

print("\nDiabetes target values after cleaning:")
print(diabetes_clean[diabetes_target].value_counts(dropna=False))


Diabetes target values after cleaning:
Outcome
0    500
1    268
Name: count, dtype: int64


In [19]:
# Save cleaned Diabetes dataset
diabetes_clean.to_csv(PROCESSED_DIR / "diabetes_cleaned.csv", index=False)
print("Saved:", PROCESSED_DIR / "diabetes_cleaned.csv")

Saved: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\processed\diabetes_cleaned.csv


## Handling Missing Values in the Diabetes Dataset

In this step, missing values are imputed using the median because all input features are numeric.

In [20]:
# Define target again if needed
diabetes_target = "Outcome"

In [21]:
# Separate input features and target
X_diabetes = diabetes_clean.drop(columns=[diabetes_target])
y_diabetes = diabetes_clean[diabetes_target]

In [23]:
# Identify numeric columns
diabetes_numeric_cols = X_diabetes.select_dtypes(include=["int64", "float64"]).columns.tolist()
print("Diabetes numeric columns:", diabetes_numeric_cols)


Diabetes numeric columns: ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']


In [24]:
# Create median imputer
diabetes_imputer = SimpleImputer(strategy="median")

In [25]:
# Apply imputation
X_diabetes[diabetes_numeric_cols] = diabetes_imputer.fit_transform(X_diabetes[diabetes_numeric_cols])


In [26]:
# Combine back with target
diabetes_imputed = pd.concat([X_diabetes, y_diabetes], axis=1)


In [27]:
# Check remaining missing values
print("\nRemaining Diabetes missing values after imputation:")
print(diabetes_imputed.isnull().sum().sort_values(ascending=False))



Remaining Diabetes missing values after imputation:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [28]:
# Preview final result
print("\nDiabetes shape after imputation:", diabetes_imputed.shape)
display(diabetes_imputed.head())


Diabetes shape after imputation: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6.0,148.0,72.0,35.0,125.0,33.6,0.627,50.0,1
1,1.0,85.0,66.0,29.0,125.0,26.6,0.351,31.0,0
2,8.0,183.0,64.0,29.0,125.0,23.3,0.672,32.0,1
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0,0
4,0.0,137.0,40.0,35.0,168.0,43.1,2.288,33.0,1


## Save the imputed diabetes data

In [29]:
# Save imputed Diabetes dataset
diabetes_imputed.to_csv(PROCESSED_DIR / "diabetes_imputed.csv", index=False)
print("Saved:", PROCESSED_DIR / "diabetes_imputed.csv")

Saved: c:\Users\adwet\Documents\Production Project\explainable-disease-risk-prediction\data\processed\diabetes_imputed.csv
